# 01. Data Cleaning and Preprocessing

## 1. Dataset Overview
The dataset contains **50,000 customer records** across **25 attributes** capturing:
- Customer demographics and location (`Age`, `Gender`, `Country`, `City`)
- Digital platform engagement and session telemetry
- Shopping, purchasing, and returns behavior
- Customer service interactions and account lifecycle metrics
- **Target Variable:** `Churned` (`0` = Retained, `1` = Churned)

---

## 2. Notebook Scope & Objectives
The primary purpose of this notebook is to clean the raw data and establish a reliable, anomaly-free baseline before exploratory analysis and modeling. 

In this notebook, we perform:
1. **Data Ingestion & Verification:** Load the dataset and inspect shapes, column types, and schemas.
2. **Missing Value Auditing:** Detect, quantify, and impute missing records using distribution-preserving strategies.
3. **Outlier & Anomaly Sanitization:** Identify and cap invalid entries (e.g., impossible ages, negative purchase counts, and percentages exceeding 100%).
4. **Data Hygiene:** Standardize feature naming conventions and verify data consistency.
5. **Clean Data Export:** Save the sanitized dataframe to `../data/cleaned_ecommerce_data.csv` for use in subsequent notebooks.

### Libraries Import & Setup


In [9]:
import pandas as pd
import numpy as np

# load raw dataset from data folder.
df = pd.read_csv("../data/ecommerce_customer_churn_dataset.csv")
print("dataset shape:",df.shape)
print("data type of dataset:\n", df.dtypes)
df.head()

dataset shape: (50000, 25)
data type of dataset:
 Age                              float64
Gender                            object
Country                           object
City                              object
Membership_Years                 float64
Login_Frequency                  float64
Session_Duration_Avg             float64
Pages_Per_Session                float64
Cart_Abandonment_Rate            float64
Wishlist_Items                   float64
Total_Purchases                  float64
Average_Order_Value              float64
Days_Since_Last_Purchase         float64
Discount_Usage_Rate              float64
Returns_Rate                     float64
Email_Open_Rate                  float64
Customer_Service_Calls           float64
Product_Reviews_Written          float64
Social_Media_Engagement_Score    float64
Mobile_App_Usage                 float64
Payment_Method_Diversity         float64
Lifetime_Value                   float64
Credit_Balance                   float64
Churned

,Age,Gender,Country,City,Membership_Years,Login_Frequency,Session_Duration_Avg,Pages_Per_Session,Cart_Abandonment_Rate,Wishlist_Items,...,Email_Open_Rate,Customer_Service_Calls,Product_Reviews_Written,Social_Media_Engagement_Score,Mobile_App_Usage,Payment_Method_Diversity,Lifetime_Value,Credit_Balance,Churned,Signup_Quarter
0,43.0,Male,France,Marseille,2.9,14.0,27.4,6.0,50.6,3.0,...,17.9,9.0,4.0,16.3,20.8,1.0,953.33,2278.0,0,Q1
1,36.0,Male,UK,Manchester,1.6,15.0,42.7,10.3,37.7,1.0,...,42.8,7.0,3.0,NaN,23.3,3.0,1067.47,3028.0,0,Q4
2,45.0,Female,Canada,Vancouver,2.9,10.0,24.8,1.6,70.9,1.0,...,0.0,4.0,1.0,NaN,8.8,NaN,1289.75,2317.0,0,Q4
3,56.0,Female,USA,New York,2.6,10.0,38.4,14.8,41.7,9.0,...,41.4,2.0,5.0,85.9,31.0,3.0,2340.92,2674.0,0,Q1
4,35.0,Male,India,Delhi,3.1,29.0,51.4,NaN,19.1,9.0,...,37.9,1.0,11.0,83.0,50.4,4.0,3041.29,5354.0,0,Q4


In [10]:
# dataset summarise and information
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 25 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Age                            47505 non-null  float64
 1   Gender                         50000 non-null  object 
 2   Country                        50000 non-null  object 
 3   City                           50000 non-null  object 
 4   Membership_Years               50000 non-null  float64
 5   Login_Frequency                50000 non-null  float64
 6   Session_Duration_Avg           46601 non-null  float64
 7   Pages_Per_Session              47000 non-null  float64
 8   Cart_Abandonment_Rate          50000 non-null  float64
 9   Wishlist_Items                 46000 non-null  float64
 10  Total_Purchases                50000 non-null  float64
 11  Average_Order_Value            50000 non-null  float64
 12  Days_Since_Last_Purchase       47000 non-null 

,Age,Membership_Years,Login_Frequency,Session_Duration_Avg,Pages_Per_Session,Cart_Abandonment_Rate,Wishlist_Items,Total_Purchases,Average_Order_Value,Days_Since_Last_Purchase,...,Returns_Rate,Email_Open_Rate,Customer_Service_Calls,Product_Reviews_Written,Social_Media_Engagement_Score,Mobile_App_Usage,Payment_Method_Diversity,Lifetime_Value,Credit_Balance,Churned
count,47505.000000,50000.000000,50000.000000,46601.000000,47000.000000,50000.000000,46000.000000,50000.000000,50000.000000,47000.000000,...,45509.000000,47472.000000,49832.000000,46500.000000,44000.000000,45000.000000,47500.000000,50000.000000,44500.000000,50000.000000
mean,37.802968,2.984009,11.624660,27.660754,8.737811,57.079973,4.298391,13.111576,123.117330,29.792872,...,6.680913,20.937980,5.681831,2.853312,29.364466,19.371607,2.353874,1440.626292,1966.233258,0.289000
std,11.834668,2.059105,7.810657,10.871013,3.778220,16.282723,3.189754,7.017312,175.569714,29.695062,...,6.143027,14.252561,2.676052,2.328948,20.574021,9.419252,1.110012,907.249443,1225.072166,0.453302
min,5.000000,0.100000,0.000000,1.000000,1.000000,0.000000,0.000000,-13.000000,26.380000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
25%,29.000000,1.400000,6.000000,19.700000,6.000000,46.400000,2.000000,8.000000,87.050000,9.000000,...,2.900000,9.900000,4.000000,1.000000,13.200000,12.500000,2.000000,789.817500,1049.000000,0.000000
50%,38.000000,2.500000,11.000000,26.800000,8.400000,58.100000,4.000000,12.000000,112.970000,21.000000,...,5.400000,19.700000,5.000000,2.000000,27.600000,18.600000,2.000000,1243.415000,1896.000000,0.000000
75%,46.000000,4.000000,17.000000,34.700000,11.200000,68.700000,6.000000,17.000000,144.440000,41.000000,...,9.100000,30.400000,7.000000,4.000000,43.100000,25.500000,3.000000,1874.000000,2791.000000,1.000000
max,200.000000,10.000000,46.000000,75.600000,24.100000,143.743350,28.000000,128.700000,9666.379178,287.000000,...,99.615734,91.700000,21.000000,21.000000,100.000000,61.900000,5.000000,8987.240000,7197.000000,1.000000


### missing value aduit

In [11]:
missing_df = pd.DataFrame({
    "missing data" : df.isnull().sum(),
    "missing percentage": (df.isnull().sum()/len(df))*100
})

missing_df

,missing data,missing percentage
Age,2495,4.990
Gender,0,0.000
Country,0,0.000
City,0,0.000
Membership_Years,0,0.000
Login_Frequency,0,0.000
Session_Duration_Avg,3399,6.798
Pages_Per_Session,3000,6.000
Cart_Abandonment_Rate,0,0.000
Wishlist_Items,4000,8.000


### Anomaly & Outlier Inspection

In [12]:
# Inspect min and max values for numerical columns to identify domain violations
anomaly_cols = ['Age', 'Total_Purchases', 'Cart_Abandonment_Rate', 'Returns_Rate']

anomaly_check = df[anomaly_cols].agg(['min', 'median', 'max']).T
anomaly_check.columns = ['Min_Value', 'Median_Value', 'Max_Value']
anomaly_check

,Min_Value,Median_Value,Max_Value
Age,5.0,38.0,200.000000
Total_Purchases,-13.0,12.0,128.700000
Cart_Abandonment_Rate,0.0,58.1,143.743350
Returns_Rate,0.0,5.4,99.615734


### 4. Domain Anomaly Sanitization (Outlier Capping)

In [13]:
# Create a copy of the raw dataframe for cleaning
df_clean = df.copy()

# Cap domain boundary violations to valid theoretical limits
df_clean['Age'] = df_clean['Age'].clip(lower=18, upper=80)
df_clean['Total_Purchases'] = df_clean['Total_Purchases'].clip(lower=0)
df_clean['Cart_Abandonment_Rate'] = df_clean['Cart_Abandonment_Rate'].clip(lower=0, upper=100)
df_clean['Returns_Rate'] = df_clean['Returns_Rate'].clip(lower=0, upper=100)

# Verify updated minimum, median, and maximum values
sanitization_summary = df_clean[['Age', 'Total_Purchases', 'Cart_Abandonment_Rate', 'Returns_Rate']].agg(['min', 'median', 'max']).T
sanitization_summary.columns = ['Sanitized_Min', 'Median', 'Sanitized_Max']
sanitization_summary

,Sanitized_Min,Median,Sanitized_Max
Age,18.0,38.0,80.000000
Total_Purchases,0.0,12.0,128.700000
Cart_Abandonment_Rate,0.0,58.1,100.000000
Returns_Rate,0.0,5.4,99.615734


### 5. Missing Value Imputation

In [14]:
# Domain-Specific Inactivity Imputation (No activity = 0)
zero_fill_cols = ['Wishlist_Items', 'Product_Reviews_Written', 'Returns_Rate']
for col in zero_fill_cols:
    df_clean[col] = df_clean[col].fillna(0.0)

# Minimum active threshold
df_clean['Payment_Method_Diversity'] = df_clean['Payment_Method_Diversity'].fillna(1.0)

# 4. Contextual Imputation for Demographics (Grouped by Country)
df_clean['Age'] = df_clean.groupby('Country')['Age'].transform(lambda x: x.fillna(x.median()))

# Add Missing Indicators for Telemetry & Engagement Features
telemetry_cols = [
    'Session_Duration_Avg', 'Pages_Per_Session', 'Social_Media_Engagement_Score',
    'Mobile_App_Usage', 'Email_Open_Rate', 'Credit_Balance', 'Days_Since_Last_Purchase', 'Discount_Usage_Rate'
]

for col in telemetry_cols:
    # Add flag so the model explicitly learns the missingness pattern
    df_clean[f'{col}_is_missing'] = df_clean[col].isnull().astype(int)
    # Impute the continuous column with median
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Handle small missing counts in Customer_Service_Calls
df_clean['Customer_Service_Calls'] = df_clean['Customer_Service_Calls'].fillna(df_clean['Customer_Service_Calls'].median())

# Verify zero remaining null values
print(f"Total Remaining Null Values: {df_clean.isnull().sum().sum()}")
print(f"Dataset Shape after adding missing flags: {df_clean.shape}")

Total Remaining Null Values: 0
Dataset Shape after adding missing flags: (50000, 33)


In [15]:
df_clean.head()

,Age,Gender,Country,City,Membership_Years,Login_Frequency,Session_Duration_Avg,Pages_Per_Session,Cart_Abandonment_Rate,Wishlist_Items,...,Churned,Signup_Quarter,Session_Duration_Avg_is_missing,Pages_Per_Session_is_missing,Social_Media_Engagement_Score_is_missing,Mobile_App_Usage_is_missing,Email_Open_Rate_is_missing,Credit_Balance_is_missing,Days_Since_Last_Purchase_is_missing,Discount_Usage_Rate_is_missing
0,43.0,Male,France,Marseille,2.9,14.0,27.4,6.0,50.6,3.0,...,0,Q1,0,0,0,0,0,0,0,0
1,36.0,Male,UK,Manchester,1.6,15.0,42.7,10.3,37.7,1.0,...,0,Q4,0,0,1,0,0,0,0,0
2,45.0,Female,Canada,Vancouver,2.9,10.0,24.8,1.6,70.9,1.0,...,0,Q4,0,0,1,0,0,0,0,0
3,56.0,Female,USA,New York,2.6,10.0,38.4,14.8,41.7,9.0,...,0,Q1,0,0,0,0,0,0,0,0
4,35.0,Male,India,Delhi,3.1,29.0,51.4,8.4,19.1,9.0,...,0,Q4,0,1,0,0,0,0,0,0


### export the clean dataset

In [16]:
df_clean.to_csv("../data/cleaned_ecommerce_data.csv" , index = False)
print("dataset save successfuly!")

dataset save successfuly!
